# Part 1 — Imports

In [1]:
from pathlib import Path
import torch

from dmpbridge.pdf.page_image_converter import convert_pdf_to_images
from dmpbridge.vision.qwen_structure_detector import detect_structure_from_images
from dmpbridge.vision.qwen_postprocessor import save_qwen_structured_blocks
from dmpbridge.processing.structure_json_builder import save_narrative_json
from dmpbridge.pdf.pdfplumber_extractor import save_pdfplumber_outputs
from dmpbridge.processing.structure_detector import detect_structure

# Part 2 — Check GPU

In [2]:
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

Torch version: 2.6.0+cu124
CUDA available: True
GPU count: 1
0 NVIDIA GeForce RTX 3090


# Part 3 — Paths

In [3]:
project_root = Path.cwd().parent

pdf_path = project_root / "data" / "raw_pdfs" / "sample4.pdf"
skeleton_path = project_root / "schemas" / "rda_dmp_dmptool_extension_skeleton.json"

qwen_output_path = project_root / "data" / "qwen_outputs" / f"{pdf_path.stem}.json"
qwen_structured_path = project_root / "data" / "qwen_outputs" / f"{pdf_path.stem}_structured_blocks.json"
qwen_json_path = project_root / "data" / "structure_json" / f"{pdf_path.stem}_qwen.json"

print("Project root:", project_root)
print("PDF path:", pdf_path)
print("PDF exists:", pdf_path.exists())
print("Skeleton exists:", skeleton_path.exists())

Project root: c:\Users\Nahid\dmpbridge
PDF path: c:\Users\Nahid\dmpbridge\data\raw_pdfs\sample4.pdf
PDF exists: True
Skeleton exists: True


# Part 4 — Convert PDF to page images

In [4]:
image_paths = convert_pdf_to_images(pdf_path, dpi=120)

print("Number of page images:", len(image_paths))

for p in image_paths:
    print(p, p.exists())

[2026-05-08 11:58:20] Converting PDF pages to images: sample4.pdf
[2026-05-08 11:58:20] Saved 3 page images to: C:\Users\Nahid\dmpbridge\data\page_images\sample4
Number of page images: 3
C:\Users\Nahid\dmpbridge\data\page_images\sample4\page_1.png True
C:\Users\Nahid\dmpbridge\data\page_images\sample4\page_2.png True
C:\Users\Nahid\dmpbridge\data\page_images\sample4\page_3.png True


# Part 5 — Run Qwen2-VL structure detection

In [5]:
qwen_results = detect_structure_from_images(
    image_paths=image_paths,
    output_path=qwen_output_path
)

print("Saved Qwen output:", qwen_output_path.exists())
print("Qwen output path:", qwen_output_path)

qwen_results

[2026-05-08 11:58:20] Loading Qwen2-VL model: Qwen/Qwen2-VL-7B-Instruct


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]

[2026-05-08 11:58:54] Running Qwen2-VL on page 1: C:\Users\Nahid\dmpbridge\data\page_images\sample4\page_1.png
[2026-05-08 12:03:11] Running Qwen2-VL on page 2: C:\Users\Nahid\dmpbridge\data\page_images\sample4\page_2.png
[2026-05-08 12:06:23] Running Qwen2-VL on page 3: C:\Users\Nahid\dmpbridge\data\page_images\sample4\page_3.png
[2026-05-08 12:10:13] Saved Qwen structure output: c:\Users\Nahid\dmpbridge\data\qwen_outputs\sample4.json
Saved Qwen output: True
Qwen output path: c:\Users\Nahid\dmpbridge\data\qwen_outputs\sample4.json


[{'document_title': 'Oxide Surfaces, From Bulk to Nanoparticles',
  'sections': [{'title': 'Types of data produced',
    'subsections': [{'title': 'Transmission electron microscopy (TEM) images'},
     {'title': 'X-ray diffraction patterns'},
     {'title': 'Sample statistics'},
     {'title': 'Spectroscopy (e.g. EDX, EELS, XPS)'}]},
   {'title': 'Quality Assurance/Control'},
   {'title': 'Data and metadata standards'}],
  'page': 1},
 {'document_title': None,
  'sections': [{'title': 'Policies for access and sharing, and provisions for appropriate protection/privacy',
    'subsections': [{'title': 'Policies for access and sharing'},
     {'title': 'Provisions for appropriate protection of privacy, confidentiality, security, intellectual property, or other rights or requirements'}]}],
  'page': 2},
 {'document_title': None,
  'sections': [{'title': 'Policies and provisions for re-use, re-distribution',
    'subsections': [{'title': 'Policies and provisions for re-use, re-distribution, 

# Part 6 — Print Qwen output clearly

In [6]:
for page in qwen_results:
    print("\nPAGE:", page.get("page"))

    if "error" in page:
        print("ERROR:", page["error"])
        print(page.get("raw_response", "")[:1000])

    for item in page.get("items", []):
        print(item.get("label"), "→", item.get("text"))


PAGE: 1

PAGE: 2

PAGE: 3


# Part 7 — Convert Qwen output to structured blocks

In [7]:
blocks = save_pdfplumber_outputs(pdf_path)
structured_blocks = detect_structure(blocks)

print("Rule-based structural labels:")

for block in structured_blocks:
    if block["label"] in ["document_title", "section", "subsection", "question"]:
        print(block["label"], "→", block["text"])

[2026-05-08 12:10:13] Extracting line-level text with pdfplumber: sample4.pdf
[2026-05-08 12:10:13] Saved line-level JSON: C:\Users\Nahid\dmpbridge\data\pdfplumber_blocks\sample4.json
[2026-05-08 12:10:13] Saved extracted text: C:\Users\Nahid\dmpbridge\data\extracted_text\sample4.txt
Rule-based structural labels:
section → 1. Transmission electron microscopy (TEM) images
section → 1. File type: .dm3, .dm4, .tiff, etc. (image files)
section → 2. Captured: Using TEMs at different facilities
section → 3. Processed: Generally using DigitalMicrograph, but also ImageJ, MacTempasX, and other
section → 1. File type: .txt
section → 2. Captured: Using X-ray diffractometers
section → 3. Processed: Will be plotted in Origin or similar data plotting software. Data analysis will
section → 3. Sample statistics
section → 1. File type: Likely spreadsheets (.xlsx)
section → 2. Captured: Will vary, likely couting statistics
section → 3. Processed: Will vary depending on what is being measured
section → 4

# Part 8 — Convert Qwen output to structured blocks

In [8]:
qwen_structured_blocks = save_qwen_structured_blocks(
    qwen_output_path=qwen_output_path,
    output_path=qwen_structured_path,
    source_pdf=pdf_path.name
)

print("Saved Qwen structured blocks:", qwen_structured_path.exists())
print("Number of Qwen structured blocks:", len(qwen_structured_blocks))

qwen_structured_blocks[:10]

[2026-05-08 12:10:13] Saved Qwen structured blocks: c:\Users\Nahid\dmpbridge\data\qwen_outputs\sample4_structured_blocks.json
Saved Qwen structured blocks: True
Number of Qwen structured blocks: 12


[{'source_pdf': 'sample4.pdf',
  'page': 1,
  'line_order': 1,
  'text': 'Oxide Surfaces, From Bulk to Nanoparticles',
  'label': 'document_title',
  'document_format': 'qwen_vl',
  'extractor': 'qwen_vl'},
 {'source_pdf': 'sample4.pdf',
  'page': 1,
  'line_order': 2,
  'text': 'Types of data produced',
  'label': 'section',
  'document_format': 'qwen_vl',
  'extractor': 'qwen_vl'},
 {'source_pdf': 'sample4.pdf',
  'page': 1,
  'line_order': 3,
  'text': 'Transmission electron microscopy (TEM) images',
  'label': 'subsection',
  'document_format': 'qwen_vl',
  'extractor': 'qwen_vl'},
 {'source_pdf': 'sample4.pdf',
  'page': 1,
  'line_order': 4,
  'text': 'X-ray diffraction patterns',
  'label': 'subsection',
  'document_format': 'qwen_vl',
  'extractor': 'qwen_vl'},
 {'source_pdf': 'sample4.pdf',
  'page': 1,
  'line_order': 5,
  'text': 'Sample statistics',
  'label': 'subsection',
  'document_format': 'qwen_vl',
  'extractor': 'qwen_vl'},
 {'source_pdf': 'sample4.pdf',
  'page': 1

# Part 9 — Build narrative JSON from Qwen blocks

In [9]:
qwen_json = save_narrative_json(
    structured_blocks=qwen_structured_blocks,
    output_path=qwen_json_path,
    skeleton_path=skeleton_path
)

print("Saved Qwen narrative JSON:", qwen_json_path.exists())
print("Qwen JSON path:", qwen_json_path)

[2026-05-08 12:10:13] Saved narrative JSON: c:\Users\Nahid\dmpbridge\data\structure_json\sample4_qwen.json
Saved Qwen narrative JSON: True
Qwen JSON path: c:\Users\Nahid\dmpbridge\data\structure_json\sample4_qwen.json


# Part 10 — Inspect Qwen narrative JSON

In [10]:
sections = qwen_json["narrative"]["template"]["section"]

print("Number of sections:", len(sections))

for section in sections:
    print(section["order"], section["title"], "| questions:", len(section["question"]))

Number of sections: 6
1 Types of data produced | questions: 4
2 Quality Assurance/Control | questions: 0
3 Data and metadata standards | questions: 0
4 Policies for access and sharing, and provisions for appropriate protection/privacy | questions: 1
5 Policies and provisions for re-use, re-distribution | questions: 0
6 Plans for archiving and preservation | questions: 0


In [11]:
sections[0] if sections else "No sections created"

{'id': 'section_1',
 'title': 'Types of data produced',
 'description': None,
 'order': 1,
 'question': [{'id': 'question_1_1',
   'text': 'Transmission electron microscopy (TEM) images',
   'order': 1,
   'answer': {'id': 'answer_1_1',
    'json': {'type': 'text',
     'answer': [{'text': ''}],
     'meta': {'schemaVersion': None}}}},
  {'id': 'question_1_2',
   'text': 'X-ray diffraction patterns',
   'order': 2,
   'answer': {'id': 'answer_1_2',
    'json': {'type': 'text',
     'answer': [{'text': ''}],
     'meta': {'schemaVersion': None}}}},
  {'id': 'question_1_3',
   'text': 'Sample statistics',
   'order': 3,
   'answer': {'id': 'answer_1_3',
    'json': {'type': 'text',
     'answer': [{'text': ''}],
     'meta': {'schemaVersion': None}}}},
  {'id': 'question_1_4',
   'text': 'Spectroscopy (e.g. EDX, EELS, XPS)',
   'order': 4,
   'answer': {'id': 'answer_1_4',
    'json': {'type': 'text',
     'answer': [{'text': ''}],
     'meta': {'schemaVersion': None}}}}]}